In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive, files
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [ ]:
drive.mount("/content/drive")
CARPETA_DRIVE = Path("/content/drive/MyDrive/ModelLOA/Datos_Procesados")
ARCHIVO_DATOS = CARPETA_DRIVE / "dataset_homicidios_final.csv"

df_homicidios = pd.read_csv(
    ARCHIVO_DATOS,
    engine="python",
    sep=None,
    encoding="utf-8-sig",
)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
df_homicidios

In [ ]:
# Convertir fecha
df_homicidios["fecha"] = pd.to_datetime(df_homicidios["fecha"], dayfirst=True)

# Variables predictoras
columnas = [
    "codigo_provincia",
    "codigo_distrito",
    "franja_horaria",
    "es_feriado",
    "anio",
    "mes",
    "dia_semana_num"
]

X = df_homicidios[columnas]
y = df_homicidios["categoria_incidencia"]

In [ ]:
# One-Hot Encoding para variables categóricas
X = pd.get_dummies(
    X,
    columns=[
        "codigo_distrito",
        "franja_horaria"
    ],
    dtype=int
)

X.head()

In [ ]:
# Fecha de corte (80% de las fechas)
fecha_corte = df_homicidios["fecha"].quantile(0.80)

# Separar entrenamiento y prueba
train_mask = df_homicidios["fecha"] < fecha_corte
test_mask = df_homicidios["fecha"] >= fecha_corte

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Fecha de corte:", fecha_corte)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
modelo = DecisionTreeClassifier(
    criterion="gini",
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced"
)

# Entrenamiento
modelo.fit(X_train, y_train)

# Predicciones
y_pred = modelo.predict(X_test)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=modelo.classes_
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=modelo.classes_,
    yticklabels=modelo.classes_
)

plt.title("Matriz de confusión - Decision Tree")
plt.xlabel("Categoría predicha")
plt.ylabel("Categoría real")

plt.show()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)

print("RESULTADOS - DECISION TREE")
print("----------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": modelo.feature_importances_
})

importancias = importancias.sort_values(
    by="Importancia",
    ascending=False
).head(15)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=importancias,
    x="Importancia",
    y="Variable"
)

plt.title("Top 15 variables más importantes - Decision Tree")
plt.xlabel("Importancia")
plt.ylabel("Variable")

plt.tight_layout()
plt.show()